# Building Defect Dataset – Cleaning
Copy the raw YOLO dataset into `data-cleaned/building-defect-detection` while skipping any images whose annotation files are empty.

In [8]:
from pathlib import Path
import ast
import shutil
from typing import Optional
from tqdm.auto import tqdm

In [ ]:
raw_root = Path("../data/data_raw/building-defect-detection-3.v3i.yolov7pytorch-001/train")
images_dir = raw_root / "images"
labels_dir = raw_root / "labels"

clean_root = Path("../data/data-cleaned/building-defect-detection")
clean_images_dir = clean_root / "images"
clean_labels_dir = clean_root / "labels"

classes_path = clean_root / "classes.txt"

clean_images_dir.mkdir(parents=True, exist_ok=True)
clean_labels_dir.mkdir(parents=True, exist_ok=True)

# Write classes.txt based on the dataset metadata.
dataset_root = raw_root.parent
data_yaml = dataset_root / "data.yaml"
names = None
if data_yaml.exists():
    for raw_line in data_yaml.read_text().splitlines():
        line = raw_line.strip()
        if line.startswith("names:"):
            names = ast.literal_eval(line.split(":", 1)[1].strip())
            break
if names:
    classes_path.write_text("\n".join(names) + "\n")

raw_root, images_dir.exists(), labels_dir.exists(), clean_root


In [10]:
def find_image_for_label(stem: str) -> Optional[Path]:
    """Return the matching image file if it exists for a given label stem."""
    for ext in (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"):
        candidate = images_dir / f"{stem}{ext}"
        if candidate.exists():
            return candidate
    return None

copied, skipped = [], []
label_files = sorted(labels_dir.glob("*.txt"))

for label_path in tqdm(label_files, desc="Copying labeled data", total=len(label_files)):
    if label_path.stat().st_size == 0:
        skipped.append(label_path.name)
        continue

    image_path = find_image_for_label(label_path.stem)
    if image_path is None:
        skipped.append(label_path.name)
        continue

    shutil.copy2(image_path, clean_images_dir / image_path.name)
    shutil.copy2(label_path, clean_labels_dir / label_path.name)
    copied.append(image_path.name)

print(f"Copied {len(copied)} labeled images.")
print(f"Skipped {len(skipped)} files with empty or missing annotations.")
print(f"Cleaned data lives in: {clean_root}")


Copying labeled data: 100%|██████████| 5104/5104 [00:17<00:00, 284.65it/s]

Copied 2144 labeled images.
Skipped 2960 files with empty or missing annotations.
Cleaned data lives in: data-cleaned/building-defect-detection


In [11]:
clean_images = sorted(clean_images_dir.glob('*'))
clean_labels = sorted(clean_labels_dir.glob('*.txt'))

image_stems = {img.stem: img for img in clean_images}
label_stems = {lbl.stem: lbl for lbl in clean_labels}

images_without_labels = sorted(stem for stem in image_stems if stem not in label_stems)
labels_without_images = sorted(stem for stem in label_stems if stem not in image_stems)

total_annotations = 0
for label_path in clean_labels:
    with label_path.open() as fh:
        total_annotations += sum(1 for line in fh if line.strip())

print(f'Cleaned images: {len(clean_images)}')
print(f'Cleaned label files: {len(clean_labels)}')
print(f'Total annotation entries: {total_annotations}')

if not images_without_labels and not labels_without_images:
    print('All image files have a corresponding label file.')
else:
    print('There are image files with missing labels or label files with missing images.')
    if images_without_labels:
        print('Images missing labels:')
        for stem in images_without_labels:
            print('  -', image_stems[stem].name)
    if labels_without_images:
        print('Labels missing images:')
        for stem in labels_without_images:
            print('  -', label_stems[stem].name)

Cleaned images: 2144
Cleaned label files: 2144
Total annotation entries: 5928
All image files have a corresponding label file.


## Refinement: Remove low-resolution images
Remove any images with the shorter side below 6000px (`min_dim` in the next cell), then re-check resolution distribution. Note this is a much stricter threshold than "1080p" — it keeps only the 8000x6000 bucket and discards everything else, including many images that would pass a typical 1080p bar.


In [12]:
from PIL import Image
import pandas as pd

def get_image_resolution(path: Path) -> tuple[int, int]:
    with Image.open(path) as img:
        return img.size  # (width, height)

def build_resolution_table(image_paths: list[Path]) -> pd.DataFrame:
    resolutions = [get_image_resolution(p) for p in image_paths]
    df = pd.DataFrame(resolutions, columns=["width", "height"])
    counts = df.value_counts().reset_index(name="count")
    return counts.sort_values(["count", "width", "height"], ascending=[False, False, False]).reset_index(drop=True)

resolution_table = build_resolution_table(sorted(clean_images_dir.glob('*')))
resolution_table


,width,height,count
0,8000,6000,1869
1,4080,3072,65
2,5280,3956,47
3,5712,4284,29
4,5472,3078,24
5,4000,3000,24
6,4032,3024,18
7,3000,4000,10
8,5472,3648,7
9,5320,4600,7


In [13]:
min_dim = 6000

removed_images = []
removed_labels = []

for image_path in sorted(clean_images_dir.glob('*')):
    width, height = get_image_resolution(image_path)
    if min(width, height) < min_dim:
        label_path = clean_labels_dir / f"{image_path.stem}.txt"
        image_path.unlink()
        removed_images.append(image_path.name)
        if label_path.exists():
            label_path.unlink()
            removed_labels.append(label_path.name)

print(f"Removed {len(removed_images)} images below {min_dim}p.")
print(f"Removed {len(removed_labels)} label files for low-resolution images.")


Removed 275 images below 6000p.
Removed 275 label files for low-resolution images.


In [14]:
resolution_table_after = build_resolution_table(sorted(clean_images_dir.glob('*')))
resolution_table_after


,width,height,count
0,8000,6000,1869
